# **1. 셀리니움**
셀레니움(Selenium)은 파이썬 코드로 실제 웹 브라우저(Chrome, Edge 등)를 자동으로 제어할 수 있게 해주는 웹 자동화 라이브러리입니다. 일반적인 requests 크롤링은 서버에서 받은 HTML만 분석하지만, Selenium은 브라우저를 직접 실행하여 버튼 클릭, 검색 입력, 스크롤, 로그인, 페이지 이동 등의 사용자 동작을 그대로 수행할 수 있기 때문에 JavaScript로 동적으로 생성되는 데이터까지 가져올 수 있습니다. 따라서 멜론, 유튜브, 쇼핑몰처럼 JavaScript 렌더링이 많은 사이트의 크롤링이나 웹 자동화 테스트에서 매우 많이 사용됩니다. 보통 webdriver.Chrome()으로 브라우저를 실행하고, find_element()로 요소를 찾으며, click(), send_keys() 등을 통해 자동화 작업을 수행합니다.

```
python -m pip install selenium
```

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "http://127.0.0.1:5500/7.html"    # 자바스크립트로 생성된 html이라서 정보를 가져올 수 없음..!! (동적)
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")
fruits = soup.select(".fruit")
print(fruits)

[]


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
import time

driver = webdriver.Chrome()     # selenium 객체 안에 Chrome() 메서드가 존재. 크롬브라우저를 컨트롤
url = "http://127.0.0.1:5500/7.html"
driver.get(url)     # 실행 시 크롬 창 띄워짐

# driver = webdriver.Chrome()
# driver.get("http://127.0.0.1:5500/7.html")

# JavaScript 실행 대기
time.sleep(2)       # 2초 대기
fruits = driver.find_elements(By.CLASS_NAME, "fruit")       # 클래스 이름으로 찾겠다는 의미

for fruit in fruits:
    print(fruit.text)

driver.quit()

사과
바나나
오렌지
딸기


> 👉 find_elements()는 Selenium에서 HTML 요소를 여러 개 찾을 때 사용하는 메서드입니다. 하나의 요소만 찾는 find_element()와 달리, 조건에 맞는 요소들을 리스트 형태로 모두 반환합니다. 예를 들어 여러 개의 이미지, 게시글, 테이블 행(tr) 등을 반복해서 크롤링할 때 매우 자주 사용됩니다. 찾는 방식은 By.ID, By.CLASS_NAME, By.CSS_SELECTOR, By.XPATH 등 다양한 선택자를 사용할 수 있으며, 반환 결과는 리스트이므로 for문으로 반복 처리하는 경우가 많습니다. 또한 find_element()는 요소를 찾지 못하면 에러가 발생하지만, find_elements()는 빈 리스트([])를 반환하기 때문에 반복 크롤링에서 더 안정적으로 사용되는 경우도 많습니다.

***
# **2. 멜론 아티스트 곡 리스트 크롤링**

### ※ xpath
XPath는 XML 또는 HTML 문서 내에서 특정 요소나 속성을 선택하기 위해 사용되는 경로 표현 언어입니다. 웹 크롤링이나 자동화 도구에서 주로 사용되며, 요소를 효율적으로 찾을 수 있도록 도와줍니다. 일반적인 XPath는 특정 위치나 속성을 기준으로 요소를 선택하는 상대적인 경로를 사용합니다. 또한 full xpath는 루트 요소에서 시작하여 대상 요소까지의 절대적인 경로를 나타냅니다. 따라서 문서 구조가 변경되면 경로가 깨질 가능성이 높습니다.

In [3]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### 페이징 없는 경우

In [ ]:

def melon_search_from_main(keyword):
    # 1. webdriver를 실행할 때, 설정할 옵션을 Options()로 생성
    options = Options()     
    options.add_argument("--start-maximized")
    # Options() : 크롭 실행 옵션을 담는 객체  
    # add_argument() : 크롬을 실행할 때 추가 명령어를 넣는 메서드
    # --start-maximized : 브라우저 시작 시 브라우저 최대화 (반응형에서는 레이아웃이 달라질 수 있기 때문에)

    # 2. webdriver로 Chrome브라우저를 컨트롤하는 Chrome()메서드에 1번에서 만든 options을 넣어줌
    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)        
    # webdriver : 특정 요소가 나타날 때까지 일정 시간 기다려주는 기능
    # WebDriverWait() : 최대 몇 초까지 기다리면서 조건이 만족되면 바로 다음 코드 실행 (안정성 up)
    # wait 변수 : 여기서 바로 실행되는 것이 아니고 아래에서 사용할 코드를 미리 지정

    data = []       # 데이터를 담을 빈 리스트 생성

    try:
        # 3. Chrome 창에서 get으로 불러온 url을 띄우고 2초 대기
        driver.get("https://www.melon.com/")
        time.sleep(2)

        # 4. id가 top_search인 것을 찾아서 이동 (검색창)
        # 못찾으면 찾을 때까지 10초 대기(wait). 나타나면 search_box에 대입
        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        # 5. 초기화하고, 검색어 입력하고 엔터키 입력하고 3초 대기
        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)
        time.sleep(3)
        
        # 6. XPATH가 정규식을 만족하는 태그를 찾아서 click 가능한 요소인지 확인 (아직 click한 건 아님)
        # (곡 페이지로 넘어가기 위해)
        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
                # <span> 우클릭 - Copy - Copy Xpath
            )
        )

        # 7. song_tab을 클릭 후 3초 대기
        song_tab.click()
        time.sleep(3)

        # 8. XPATH가 정규식을 만족하는 태그를 찾아서 이동
        # (곡 목록이 있는 테이블로 이동)
        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )

        # 9. song_table에서 CSS_SELECTOR로 tbody의 자손 중 tr 태그를 모두 선택(여러 개)해서 길이 출력
        rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")
        print("찾은 행 개수:", len(rows))

        # 10. 여러 행들 중 하나씩 뽑아서 for문 반복
        for row in rows:
            try:
                #11. 태그가 td인 요소를 찾아서 cols에 저장하고 그 길이가 5보다 작으면 게속 진행 ❓
                cols = row.find_elements(By.TAG_NAME, "td")
                if len(cols) < 5:
                    continue

                # [곡명 td]
                # 12. 곡명이 담긴 요소의 텍스트를 공백 제거하여 title_text에 저장
                title_text = cols[2].text.strip()       # cols의 2번 인덱스가 곡명을 포함
                title_lines = [
                    line.strip()        # ❓
                    for line in title_text.split("\n")
                    if line.strip()     # line.strip()해도 데이터가 있다면
                ]

                title = ""

                for line in title_lines:
                    if (
                        "재생" not in line
                        and "담기" not in line
                        and "상세정보" not in line
                        and not line.startswith("Title")
                    ):
                        title = line
                        break

                # Title 줄이 더 정확한 경우 보정
                for line in title_lines:
                    if line.startswith("Title "):
                        title = line.replace("Title ", "").strip()
                        break

                # 아티스트 td
                artist_text = cols[3].text.strip()      # 인덱스 3번째 td
                artist_lines = [
                    line.strip()
                    for line in artist_text.split("\n") # 아티스트가 여러 명인 경우 처리를 위함 
                    if line.strip()
                ]
                artist = artist_lines[0] if artist_lines else ""

                # 앨범 td
                album_text = cols[4].text.strip()
                album_lines = [
                    line.strip()
                    for line in album_text.split("\n")
                    if line.strip()
                ]
                album = album_lines[0] if album_lines else ""

                # 좋아요 수 td
                like = ""
                try:
                    like = row.find_element(
                        By.CSS_SELECTOR,
                        "button.like span.cnt"
                        # #frm_defaultList > div > table > tbody > tr:nth-child(1) > td:nth-child(6) > div > button > span.cnt > span
                    ).text.strip()
                except:
                    pass

                if title:
                    data.append({
                        "곡명": title,
                        "아티스트": artist,
                        "앨범": album,
                        "좋아요수": like
                    })

            except Exception as e:
                print("행 처리 오류:", e)

        # 데이터프레임 생성
        df = pd.DataFrame(data)

        # df가 비어있지 않으면 idx + 1 (인덱스가 1부터 출력되도록)
        if not df.empty:
            df.index = df.index + 1

        file_name = f"melon_{keyword}_songs.csv"
        df.to_csv(file_name, encoding="utf-8-sig")

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()
      

In [5]:
melon_search_from_main("신의키스")

찾은 행 개수: 6
CSV 저장 완료: melon_신의키스_songs.csv
총 6곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,이젠 내가 더 사랑해 (PJ1),신의키스,The Bucket List,5
2,헤어질 준비,신의키스,The Bucket List,5
3,너 없는 생일,신의키스,The Bucket List,4
4,버킷 리스트,신의키스,The Bucket List,5
5,서로를 가장 잘 아는 남,신의키스,The Bucket List,6
6,이젠 내가 더 사랑해 (PJ2),신의키스,The Bucket List,4


***

### 페이징 있는 경우

In [13]:
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException         # 추가

In [14]:
def extract_current_page(driver, wait):
    data = []
    try:
        song_table = wait.until(
            EC.presence_of_element_located(
                (By.XPATH, '//*[@id="frm_defaultList"]/div/table')
            )
        )
    except TimeoutException:
        return []
    
    rows = song_table.find_elements(By.CSS_SELECTOR, "tbody tr")        # 50줄

    for row in rows:
        cols = row.find_elements(By.TAG_NAME, "td")

        if len(cols) < 5:       # td가 5개 안되면 continue
            continue

        title_lines = [
            line.strip()
            for line in cols[2].text.split("\n")        # title에서 제목만 얻어서 리스트에 넣어주도록
            if line.strip()
        ]
        title = ""

        for line in title_lines:
            if line.startswith("Title "):
                title = line.replace("Title ", "").strip()      # "Title: " 문자열 지우고
                break

        if not title:       # title이 없다면
            for line in title_lines:
                if (
                    "재생" not in line
                    and "담기" not in line
                    and "상세정보" not in line
                    and not line.startswith("Title")
                ):
                    title = line
                    break

        artist_lines = [
            line.strip()
            for line in cols[3].text.split("\n")        # 여러명일 경우 아티스트 가져와서 처리
            if line.strip()
        ]
        artist = artist_lines[0] if artist_lines else ""

        album_lines = [
            line.strip()
            for line in cols[4].text.split("\n")
            if line.strip()
        ]
        album = album_lines[0] if album_lines else ""

        try:
            like = row.find_element(
                By.CSS_SELECTOR,
                "button.like span.cnt"
            ).text.strip()
        except:
            like = ""

        if title:
            data.append({
                "곡명": title,
                "아티스트": artist,
                "앨범": album,
                "좋아요수": like
            })

    return data

In [15]:
# 이어서 페이징 처리
def melon_search_all_pages(keyword, max_page=30):       # 페이지가 더 있다면 30보다 큰 수로 설정
    options = Options()
    options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, 10)

    all_data = []

    try:
        driver.get("https://www.melon.com/")
        time.sleep(2)

        search_box = wait.until(
            EC.presence_of_element_located((By.ID, "top_search"))
        )

        search_box.clear()
        search_box.send_keys(keyword)
        search_box.send_keys(Keys.ENTER)
        time.sleep(3)

        song_tab = wait.until(
            EC.element_to_be_clickable(
                (By.XPATH, '//*[@id="divCollection"]/ul/li[3]/a/span')
            )
        )
        song_tab.click()
        time.sleep(3)

        for page in range(1, max_page + 1):
            page_data = extract_current_page(driver, wait)  # 크롬 객체와 wait를 전달

            if not page_data:       # 페이지 데이터가 없는지 검사
                print(f"{page}페이지 데이터가 없어 종료합니다.")
                break

            all_data.extend(page_data)
            print(f"{page}페이지 크롤링 완료: {len(page_data)}곡")

            next_start_index = page * 50 + 1

            # ⭐
            try:
                driver.execute_script(      # execute_script(): 자바스크립트를 실행하도록 해주는 메서드
                    f"pageObj.sendPage('{next_start_index}');"  # 2페이지에 커서를 hover하면 화면 하단에 자바스크립트 코드가 뜸 (그 자바스크립트를 실행해서 이동하는 것임)
                )
                time.sleep(3)

            except Exception as e:
                print("다음 페이지 이동 실패:", e)
                break

        df = pd.DataFrame(all_data)

        if not df.empty:
            df = df.drop_duplicates(
                subset=["곡명", "아티스트", "앨범"]
            )
            df.index = df.index + 1

        file_name = f"melon_{keyword}_all_songs.csv"

        df.to_csv(
            file_name,
            encoding="utf-8-sig"
        )

        print(f"CSV 저장 완료: {file_name}")
        print(f"총 {len(df)}곡 수집 완료")

        return df

    finally:
        driver.quit()

In [16]:
melon_search_all_pages("조용필", 30)

1페이지 크롤링 완료: 50곡
2페이지 크롤링 완료: 50곡
3페이지 크롤링 완료: 50곡
4페이지 크롤링 완료: 50곡
5페이지 크롤링 완료: 50곡
6페이지 크롤링 완료: 50곡
7페이지 크롤링 완료: 50곡
8페이지 크롤링 완료: 50곡
9페이지 크롤링 완료: 50곡
10페이지 크롤링 완료: 50곡
11페이지 크롤링 완료: 50곡
12페이지 크롤링 완료: 50곡
13페이지 크롤링 완료: 50곡
14페이지 크롤링 완료: 50곡
15페이지 크롤링 완료: 50곡
16페이지 크롤링 완료: 50곡
17페이지 크롤링 완료: 50곡
18페이지 크롤링 완료: 50곡
19페이지 크롤링 완료: 50곡
20페이지 크롤링 완료: 50곡
21페이지 크롤링 완료: 50곡
22페이지 크롤링 완료: 50곡
23페이지 크롤링 완료: 50곡
24페이지 크롤링 완료: 50곡
25페이지 크롤링 완료: 50곡
26페이지 크롤링 완료: 20곡
27페이지 데이터가 없어 종료합니다.
CSV 저장 완료: melon_조용필_all_songs.csv
총 1270곡 수집 완료


,곡명,아티스트,앨범,좋아요수
1,바람의 노래,조용필,조용필 16집,"26,576"
2,꿈,조용필,The Dreams,"16,275"
3,HOT 잊혀진 사랑,조용필,30주년 기념 음반 Part 1,"2,414"
4,Bounce,조용필,Hello,"40,021"
5,HOT 단발머리,조용필,조용필 1집 (Remastered),"11,088"
...,...,...,...,...
1266,님이여,조용필,기쁜 우리 젊은날의 가요 4집,15
1267,마음속의 그림자,조용필,기쁜 우리 젊은날의 가요 4집,14
1268,세월이 가면,조용필,기쁜 우리 젊은날의 가요 1집,24
1269,Bounce - 조용필 (MR),음악상자,음악상자 12집,3


***
# **3. 스타벅스 서울 전체 매장 크롤링**

In [24]:
import re
from selenium.webdriver import ActionChains     # 드래그해서 이동 (액션을 연결해서 사용했다~)
from bs4 import BeautifulSoup

def fetch_starbucks():
    url = "https://www.starbucks.co.kr/index.do"
    driver = webdriver.Chrome()
    driver.maximize_window()            # 옵션으로 줬던거랑 똑같음
    driver.get(url)
    time.sleep(2)

    # 메뉴 이동
    action = ActionChains(driver)       # 객체 생성

    first_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03"
    )

    second_tag = driver.find_element(
        By.CSS_SELECTOR,
        "#gnb > div > nav > div > ul > li.gnb_nav03 > div > div > div > ul:nth-child(1) > li:nth-child(3) > a"
    )
    #gnb > div > nav > div > ul > li.gnb_nav03 > div > div > div > ul:nth-child(1) > li.gnb_sub_ttl > a

    # \는 연결되어있다고 생각하면 됨
    # first_tag로 가고 second_tag로 가서 click!
    # perform() 여기까지 실행
    action.move_to_element(first_tag) \
          .move_to_element(second_tag) \
          .click() \
          .perform()

    # 서울 선택
    seoul_tag = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((
            By.CSS_SELECTOR,
            "#container > div > form > fieldset > div > section > article.find_store_cont > article > article:nth-child(4) > div.loca_step1 > div.loca_step1_cont > ul > li:nth-child(1) > a"
        ))
    )
    seoul_tag.click()

    # 구 목록 로딩 대기
    # 클래스가 동일해도 인덱스를 이용해서 "전체"를 선택할 수 있음 (selector로 선택해도 무관)
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "set_gugun_cd_btn")
        )
    )

    gu_elements = driver.find_elements(
        By.CLASS_NAME,
        "set_gugun_cd_btn"
    )

    # 전체 선택
    gu_elements[0].click()

    # 매장 목록 로딩 대기
    WebDriverWait(driver, 5).until(
        EC.presence_of_all_elements_located(
            (By.CLASS_NAME, "quickResultLstCon")
        )
    )

    # HTML 가져오기
    req = driver.page_source        # driver.page_source : 현재 상황의 HTML 소스를 req에 저장 (req은 HTML이 저장되어 있음)
    soup = BeautifulSoup(req, "html.parser")    # html 파서를 이용해 soup 객체 생성

    stores = soup.find(
        'ul',
        'quickSearchResultBoxSidoGugun'
    ).find_all('li')

    # 데이터 저장 리스트
    store_list = []
    addr_list = []
    lat_list = []
    lng_list = []

    # 데이터 추출
    for store in stores:
        store_name = store.find("strong").text
        store_addr = store.find("p").text

        # 전화번호 제거
        store_addr = re.sub(
            r'\d{4}-\d{4}$',        # 정규식 r데이터? \d를 숫자로 인식 (숫자4자-숫자4자 를 찾아서 지우기)
            '',
            store_addr
        ).strip()

        store_lat = store['data-lat']
        store_lng = store['data-long']

        store_list.append(store_name)
        addr_list.append(store_addr)
        lat_list.append(store_lat)
        lng_list.append(store_lng)

    # 데이터프레임 생성
    df = pd.DataFrame({
        'store': store_list,
        'addr': addr_list,
        'lat': lat_list,
        'lng': lng_list
    })

    driver.quit()

    return df

In [26]:
# 함수 실행
starbucks_df = fetch_starbucks()

# CSV 저장
starbucks_df.to_csv(
    "starbucks_seoul.csv",
    # index=False,
    encoding='utf-8-sig'
)

print("데이터가 starbucks_seoul.csv 파일로 저장되었습니다.")
print(starbucks_df.head())

데이터가 starbucks_seoul.csv 파일로 저장되었습니다.
       store                        addr         lat          lng
0  역삼아레나빌딩       서울특별시 강남구 언주로 425 (역삼동)   37.501087   127.043069
1   논현역사거리      서울특별시 강남구 강남대로 538 (논현동)   37.510178   127.022223
2  신사역성일빌딩      서울특별시 강남구 강남대로 584 (논현동)  37.5139309  127.0206057
3   국기원사거리      서울특별시 강남구 테헤란로 125 (역삼동)   37.499517   127.031495
4   대치재경빌딩    서울특별시 강남구 남부순환로 2947 (대치동)   37.494668   127.062583
